# XAI TCNER Project

## Train Base Model

In [ ]:
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Load spaCy and define preprocessing
# ----------------------------
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and len(token) > 2
    ]
    return " ".join(tokens)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv("DBLPTrainset.txt", sep="\t", header=None, names=["Label", "Title"])
test_df = pd.read_csv("DBLPTestset.txt", sep="\t", header=None, names=["Title"])
test_truth = pd.read_csv("DBLPTestGroundTruth.txt", sep="\t", header=None, names=["Label"])
y_test = test_truth["Label"]

# ----------------------------
# Preprocess titles
# ----------------------------
train_df["Cleaned_Title"] = train_df["Title"].apply(preprocess)
test_df["Cleaned_Title"] = test_df["Title"].apply(preprocess)

y_train = train_df["Label"]
y_test = test_truth["Label"]

print("Class Distribution in Training Set:")
print(y_train.value_counts())
print("\nClass Distribution in Test Set:")
print(y_test.value_counts())

# ----------------------------
# Data Quality Checks
# ----------------------------
train_dup_titles = train_df.duplicated(subset="Title").sum()
test_dup_titles = test_df.duplicated(subset="Title").sum()
unique_train_dup_titles = train_df[train_df.duplicated(subset="Title", keep=False)]["Title"].nunique()
train_full_row_dups = train_df.duplicated().sum()
test_full_row_dups = test_df.duplicated().sum()

print("Train duplicate titles:", train_dup_titles)
print("Unique duplicate titles in train set:", unique_train_dup_titles)
print("Train full-row duplicates:", train_full_row_dups)
print("Test duplicate titles:", test_dup_titles)
print("Test full-row duplicates:", test_full_row_dups)

train_df = train_df.drop_duplicates(subset="Title", keep="first")

print("\nMissing values in training set:")
print(train_df.isnull().sum())
print("\nMissing values in test set:")
print(test_df.isnull().sum())
print("\nMissing values in test ground truth:")
print(test_truth.isnull().sum())

# ----------------------------
# Outlier Detection (Title Length)
# ----------------------------
train_df["title_char_len"] = train_df["Title"].apply(len)
test_df["title_char_len"] = test_df["Title"].apply(len)

print("\nOutlier Check - Title Lengths:")
print("Training set → Min:", train_df["title_char_len"].min(), "| Max:", train_df["title_char_len"].max())
print("Test set     → Min:", test_df["title_char_len"].min(), "| Max:", test_df["title_char_len"].max())

# ----------------------------
# TF-IDF vectorization
# ----------------------------
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=3,
    max_df=0.85
)

X_train = vectorizer.fit_transform(train_df["Cleaned_Title"])
X_test = vectorizer.transform(test_df["Cleaned_Title"])
y_train = train_df["Label"]

# ----------------------------
# Train model: Logistic Regression 
# ----------------------------
print("\nTraining LogisticRegression...")
model = LogisticRegression(max_iter=1000, solver='liblinear', multi_class='ovr')
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluation
print("\n=== FINAL TEST SET EVALUATION ===")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)

## LOCAL - Lime Explanations

In [ ]:
import random
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline

In [ ]:
# ----------------------------
# LIME Explanations for 10 Random Samples
# ----------------------------
print("\n=== LIME Explanations for 10 Random Samples ===")
pipeline = make_pipeline(vectorizer, model)
class_names = sorted(y_train.unique())
explainer = LimeTextExplainer(class_names=class_names)

# Choose 10 random indices from the test set
random_indices = random.sample(range(len(test_df)), 10)

for idx in random_indices:
    sample_text = test_df.iloc[idx]["Title"]
    true_label = y_test.iloc[idx]

    print(f"\n--- Sample Index: {idx} ---")
    print("Selected Title:", sample_text)
    print("True Label:", true_label)

    exp = explainer.explain_instance(sample_text, pipeline.predict_proba, num_features=10)

    print("Top Features from LIME:")
    for feature, weight in exp.as_list():
        print(f"{feature}: {weight:.4f}")

    # Optional: show HTML explanation in Jupyter
    exp.show_in_notebook(text=sample_text)

## LOCAL - SHAP Explanations (Misclassified Instances)

In [ ]:
import shap
import numpy as np
from IPython.display import display
shap.initjs()

X_test_dense = X_test.toarray()
y_pred = model.predict(X_test)

misclassified_indices = np.where(y_pred != y_test)[0]
misclassified_indices = misclassified_indices[:5]

explainer = shap.Explainer(model, X_train[:100])
class_names = model.classes_

for idx in misclassified_indices:
    sample_dense = X_test_dense[idx].reshape(1, -1)
    shap_values = explainer(sample_dense)

    title = test_df.iloc[idx]["Title"]
    true_label = y_test.iloc[idx]
    pred_label = y_pred[idx]

    true_index = list(class_names).index(true_label)
    pred_index = list(class_names).index(pred_label)

    print(f"\n=== Misclassified Sample {idx} ===")
    print(f"Title           : {title}")
    print(f"True Label      : {true_label}")
    print(f"Predicted Label : {pred_label}")

    print("\n--> SHAP Explanation for Predicted Label")
    display(shap.force_plot(
        base_value=explainer.expected_value[pred_index],
        shap_values=shap_values.values[0][:, pred_index],
        features=sample_dense[0],
        feature_names=vectorizer.get_feature_names_out()
    ))

    print("\n--> SHAP Explanation for True Label")
    display(shap.force_plot(
        base_value=explainer.expected_value[true_index],
        shap_values=shap_values.values[0][:, true_index],
        features=sample_dense[0],
        feature_names=vectorizer.get_feature_names_out()
    ))

## LOCAL: Anchors (Misclassified Samples)

In [ ]:
import numpy as np
import random
import spacy
from alibi.explainers import AnchorText
from alibi.utils.lang_model import DistilbertBaseUncased

print("\n=== Anchor Explanations for 5 Random Misclassified Samples ===")

nlp = spacy.load("en_core_web_sm")
language_model = DistilbertBaseUncased()

def predict_fn(texts):
    return model.predict(vectorizer.transform(texts))

anchor_explainer = AnchorText(predict_fn, sampling_strategy="language_model", nlp=nlp, language_model=language_model)

X_test_dense = X_test.toarray()
y_pred = model.predict(X_test)
misclassified = np.where(y_pred != y_test)[0][:5]

for idx in misclassified:
    text = test_df.iloc[idx]["Title"]
    true = y_test.iloc[idx]
    pred = predict_fn([text])[0]
    exp = anchor_explainer.explain(text)
    print(f"\n=== Sample Index: {idx} ===")
    print("Title:", text)
    print("True Label:", true)
    print("Predicted Label:", pred)
    print("Anchor Rule:", " AND ".join(f'"{w}"' for w in exp.anchor))
    print("Precision:", round(exp.precision, 2))
    print("Coverage :", round(exp.coverage, 2))
    covered = exp.raw["examples"][-1]["covered_true"][:3]
    for ex in covered:
        print("-", ex)

## GLOBAL - Top Positively and Negatively Weighted Features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

feature_names = vectorizer.get_feature_names_out()
class_labels = model.classes_
top_n = 10  # number of features to show in each direction

for i, class_label in enumerate(class_labels):
    coefs = model.coef_[i]
    
    top_pos_idx = np.argsort(coefs)[-top_n:]
    top_neg_idx = np.argsort(coefs)[:top_n]

    top_pos_features = [(feature_names[j], coefs[j]) for j in top_pos_idx]
    top_neg_features = [(feature_names[j], coefs[j]) for j in top_neg_idx]

    # Combine and sort for visual clarity
    combined = top_neg_features + top_pos_features
    combined_words = [w for w, _ in combined]
    combined_weights = [w for _, w in combined]

    colors = ['red'] * top_n + ['green'] * top_n

    plt.figure(figsize=(10, 6))
    bars = plt.barh(combined_words, combined_weights, color=colors)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.title(f"Top Positive & Negative Features for Class: {class_label}", fontsize=14)
    plt.xlabel("Coefficient Weight", fontsize=12)
    plt.ylabel("Feature (TF-IDF Term)", fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


## GLOBAL - Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

X_test_dense = X_test.toarray()

results = permutation_importance(
    model,
    X_test_dense,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    max_samples=500
)

importances = results.importances_mean
stds = results.importances_std
indices = np.argsort(importances)[::-1]
feature_names = vectorizer.get_feature_names_out()

top_n = 20
print(f"\nTop {top_n} features by permutation importance:")
for idx in indices[:top_n]:
    print(f"{feature_names[idx]:<20} "
          f"{importances[idx]:.4f} +/- {stds[idx]:.4f}")

plt.figure(figsize=(8, 6))
plt.barh(
    feature_names[indices[:top_n]][::-1],
    importances[indices[:top_n]][::-1]
)
plt.xlabel("Permutation Importance")
plt.tight_layout()
plt.show()

## GLOBAL: Surrogate Model (Decision Tree)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

surrogate = DecisionTreeClassifier(max_depth=3, random_state=42)
surrogate.fit(X_train, model.predict(X_train))

plt.figure(figsize=(20, 10))
plot_tree(
    surrogate,
    feature_names=vectorizer.get_feature_names_out(),
    class_names=model.classes_,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()